# Fundamentals 00.3 - Runtime OpenAI Provider API

Objetivo: probar la ruta `openai-runtime` de forma aislada antes de construir workflows mas grandes.

Este notebook usa el provider nativo de Agentic Systems. No usa `openai-agents`.

Regla de diseno:

```text
Agentic Systems define el contrato de ejecucion.
openai-runtime define el backend OpenAI directo.
Integrations no son obligatorias para usar OpenAI.
```


## 0) Imports minimos

El notebook asume que `agentic-systems` esta instalado en el ambiente activo.


In [ ]:
import json
import os

import agentic_systems as toolkit

print("agentic_systems:", toolkit.__name__)


## Parametros de `RunPolicy`

`RunPolicy` declara c?mo debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | N?mero m?ximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | N?mero m?ximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | L?mite de tokens del modelo cuando el provider lo soporta. | ?til en providers LM; puede quedar `None` en `python-direct`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion autom?tica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | M?ximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso.


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico")


## 1) Utilidad segura de impresi?n


In [ ]:
def show_json(obj, title: str | None = None) -> None:
    if title:
        print(f"\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


## 2) RuntimeConfig y SchedulerConfig para OpenAI

Esta celda no llama a OpenAI. `toolkit.runtime(...)` lee la configuracion OpenAI del ambiente o `.env` igual que Bedrock lee su configuracion: modelo, base URL, proyecto y presencia de API key quedan visibles sin exponer secretos.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=1,
    max_tool_calls=5,
    max_turns=6,
    max_concurrency=1,
    backoff_s=0.2,
)

runtime = toolkit.runtime(
    provider="openai-runtime",
    region=None,
    scheduler=scheduler,
    metadata={"purpose": "fundamentals_openai_provider_notebook"},
)

show_json(runtime.describe(), "OpenAI runtime describe")


## 3) Configuraci?n OpenAI segura

Esta celda no pide ni guarda secretos. `openai-runtime` y `provider="auto"` leen `OPENAI_API_KEY` y la configuracion OpenAI desde el ambiente del kernel o `.env`.

Si `has_openai_api_key` es `False`, configura la variable fuera del notebook y reinicia/recarga el kernel.


In [ ]:
RUN_OPENAI_SMOKE_TESTS = bool(os.getenv("OPENAI_API_KEY"))

show_json(
    {
        "model": runtime.model_id,
        "openai_configuration": runtime.describe().get("configuration", {}).get("openai", {}),
        "run_openai_smoke_tests": RUN_OPENAI_SMOKE_TESTS,
        "has_openai_api_key": bool(os.getenv("OPENAI_API_KEY")),
        "auto_runtime_selection": toolkit.runtime(provider="auto", scheduler=scheduler).describe(),
    },
    "openai config",
)


## 4) Definir una tool local para el provider OpenAI

`openai-runtime` necesita tools concretas para ejecutar el loop nativo de tool calling. La tool sigue siendo una tool normal de Agentic Systems.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


agent = toolkit.agent(
    name="openai_runtime_smoke_agent",
    instructions="Usa la tool disponible y devuelve una respuesta breve con evidencia.",
    tools=[sumar],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["sumar"], completion="when_required_tools_satisfied"),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=3, temperature=0.0),
)

toolkit.show(agent.info(), title="OpenAI runtime agent")


## 5) Smoke opcional con `openai-runtime`

Esta celda ejecuta el provider nativo de Agentic Systems cuando hay API key. Si no hay credenciales, el notebook sigue siendo ejecutable y explica que falta.


In [ ]:
if RUN_OPENAI_SMOKE_TESTS:
    result = agent.run("Suma 10 y 20 usando la tool sumar.", mode="eval")
    toolkit.human_result(result, pretty=False, show_lineage=True)
    toolkit.show(result.normalized(), title="OpenAI runtime normalized result")
else:
    print("Saltado: configura OPENAI_API_KEY en el ambiente del kernel para ejecutar openai-runtime.")


## 6) Variantes de `mode` para ejecucion evaluable

`mode` declara la intenci?n del run. No cambia la API publica del agente, pero s? resuelve una `RunPolicy` distinta y queda registrada en el `RunResult`.

En tutorials conviene escribirlo expl?citamente porque estamos ensenando una ejecucion evaluable, no una llamada casual.


<!-- run-policy-parameters -->
## C?mo leer `RunPolicy`

`RunPolicy` es la pol?tica de ejecucion de un agente. No define que hace el negocio; define cu?nto puede intentar el agente, c?mo usa tools y que tan estricta debe ser la validaci?n.

| Parametro | Que controla | Regla pr?ctica |
| --- | --- | --- |
| `max_turns` | N?mero m?ximo de turnos del loop agente/modelo/tool. | S?belo si el agente necesita varios pasos; b?jalo para cortar loops largos. |
| `max_tool_calls` | L?mite total de ejecuciones de tools. `None` significa que no agrega un l?mite propio. | ?salo cuando el contrato exige una cantidad acotada de acciones. |
| `max_tokens` | Presupuesto de generaci?n que se pasa al provider cuando el backend lo soporta. | D?jalo en `None` para usar el default del runtime/provider. |
| `temperature` | Aleatoriedad de salida en providers LM. | `0.0` para evals reproducibles; `None` para usar el default del provider. |
| `tool_choice` | Preferencia de uso de tools en runtimes LM. | `auto` deja que el modelo decida; valores forzados dependen del provider. |
| `repair` | Permite reparar/reintentar llamadas invalidas de tools. | `True` para robustez; `False` para detectar fallos temprano. |
| `max_repairs` | N?mero m?ximo de reparaciones permitidas. | Mantener bajo en producci?n para evitar ciclos opacos. |
| `finalize` | Cu?ndo sintetizar respuesta final. | `on_max_turns` cierra al agotar turnos; `after_required_tools` cierra al cumplir contrato; `never` no sintetiza. |
| `trace` | Nivel de traza guardada. | `compact` para notebooks; `full` para auditor?a profunda. |
| `strict` | Severidad de validaci?n de contrato/policy. | `True` para producci?n/evals; `False` solo para debug exploratorio. |

Los `mode` (`default`, `fast`, `eval`, `audit`, `debug`, `prod`) son presets de `RunPolicy`. El notebook los muestra como JSON para inspecci?n, pero esta tabla explica c?mo interpretar cada campo.


In [ ]:
run_modes = ["default", "fast", "eval", "audit", "debug", "prod"]

mode_variants = {
    mode: toolkit.RunPolicy.for_mode(mode).model_dump(mode="json")
    for mode in run_modes
}

toolkit.show(mode_variants, title="Run modes -> RunPolicy")


## 7) Patron recomendado

Para este notebook usamos `mode="eval"` porque queremos una corrida reproducible y validable. En producci?n cambia la intenci?n, no la fachada:

```python
agent.run(prompt, mode="eval")   # tutorial, smoke, evaluaci?n
agent.run(prompt, mode="prod")   # producci?n conservadora
agent.run(prompt, mode="debug")  # diagnostico con traza mas amplia
```

Si no escribes `mode`, Agentic Systems usa `default`.


## 8) Lectura correcta del diseno

- `openai-runtime` es provider/backend directo.
- No depende de `openai-agents`.
- La API publica sigue siendo `toolkit.runtime(...)`, `toolkit.agent(...)`, `toolkit.tool(...)` y `toolkit.human_result(...)`.
- `runtime.describe()` es la forma estable de auditar seleccion de backend.


## Coverage API de este notebook


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='openai-runtime')", "description": "Declara OpenAI Runtime como backend canonico."},
    {"api": "RunPolicy.for_mode", "description": "Explica las variantes de mode sin ejecutar llamadas extra al provider."},
    {"api": "agent.run(..., mode='eval')", "description": "Declara una corrida evaluable y reproducible para tutorials."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion antes de llamar al provider."},
    {"api": "RuntimeConfig.describe", "description": "Expone provider, modelo y scheduler sin ejecutar inferencia."},
    {"api": "toolkit.tool", "description": "Define una tool local compatible con tool calling."},
    {"api": "toolkit.agent", "description": "Crea un agente con runtime OpenAI nativo."},
    {"api": "toolkit.human_result", "description": "Renderiza el resultado cuando el smoke test esta activo."},
    {"api": "shared scenario declared", "description": "Mantiene el mismo problema de fundamentals para comparacion 1:1."},
]

toolkit.show({"notebook": "00_runtime_openai_provider_api.ipynb", "api_coverage": api_coverage})


## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `toolkit.runtime(provider="openai-runtime")`: Runtime explicito para OpenAI nativo.
- `OPENAI_RUNTIME_ENGINE`: Nombre canonico del engine OpenAI.
- `toolkit.agent`: Fachada publica para ejecutar con runtime OpenAI.
- `toolkit.tool`: Tool local conectada al agente OpenAI.
- `AgentContract / RunPolicy`: Contrato minimo para ejecucion evaluable.
- `mask_sensitive`: Utilidad publica para no imprimir secretos.
- `human_result`: Render humano del resultado real del runtime.

